Inistaçao do ambiente java

In [ ]:
%%bash

# instalar java
apt-get update --fix-missing
apt-get install -y openjdk-17-jdk-headless -qq

# Instalação PySpark
pip install -q pyspark

Get:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Hit:2 https://cli.github.com/packages stable InRelease
Get:3 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.3 MB]
Get:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:9 http://security.ubuntu.com/ubuntu noble-security/restricted amd64 Packages [2,069 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble/main amd64 Packages [52.1 kB]
Get:11 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:12 http://archive.ubuntu.com/ubuntu noble-updates/multiverse amd64 Packages [67.4 kB]
Get:13 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 Pa

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
import os
from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import col, sum, avg, count, max, min, round as spark_round

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

spark = SparkSession.builder \
    .master('local[*]') \
    .appName("Aula_Spark_unipe") \
    .getOrCreate()

spark.conf.set("spark.sql.repl.eagerEval.enable", True)

print(f"Spark version: {spark.version}")

criando um dataframe

In [ ]:
## ORGANIZA OS ARQUIVOS NA LANDING PAGE
import os, shutil

BATCH = "lote1"
LANDING = f"/content/lakehouse/landing/mercadonordeste/{BATCH}"
os.makedirs(LANDING, exist_ok=True)

for nome in ("clientes.csv", "produtos.csv", "pedidos.jsonl", "avaliacoes.csv"):
    origem = f"/content/{nome}"
    if os.path.exists(origem):
        shutil.move(origem, f"{LANDING}/{nome}")

print(sorted(os.listdir(LANDING)))

['avaliacoes.csv', 'clientes.csv', 'pedidos.jsonl', 'produtos.csv']


In [ ]:
#lendo os arquivos csv com spark clientes.csv
LANDING = f"/content/lakehouse/landing/mercadonordeste/{BATCH}"

clientes = spark.read.csv(f"{LANDING}/clientes.csv", header=True, inferSchema=True)
clientes.printSchema()

root
 |-- cliente_id: integer (nullable = true)
 |-- nome: string (nullable = true)
 |-- email: string (nullable = true)
 |-- cidade: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- data_cadastro: string (nullable = true)
 |-- canal_aquisicao: string (nullable = true)



In [ ]:
# lendo registros do tipo Json
pedidos = spark.read.json(f"{LANDING}/pedidos.jsonl")
pedidos.printSchema()

root
 |-- cliente_id: long (nullable = true)
 |-- data_pedido: string (nullable = true)
 |-- forma_pagamento: string (nullable = true)
 |-- itens: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- preco_unit: double (nullable = true)
 |    |    |-- produto_id: long (nullable = true)
 |    |    |-- qtd: long (nullable = true)
 |-- pedido_id: long (nullable = true)
 |-- status: string (nullable = true)
 |-- valor_frete: double (nullable = true)



In [ ]:
pedidos.show()

+----------+-------------------+---------------+--------------------+---------+---------+-----------+
|cliente_id|        data_pedido|forma_pagamento|               itens|pedido_id|   status|valor_frete|
+----------+-------------------+---------------+--------------------+---------+---------+-----------+
|      2769|2025-11-28T17:05:06|            pix|    [{211.62, 6, 2}]|   123912|devolvido|       24.9|
|      3227|2025-12-22T08:51:13|  cartao_debito|   [{824.72, 39, 1}]|   106631|  enviado|        0.0|
|      4575|2026-02-10T01:37:50|            pix|[{220.7, 82, 3}, ...|   127545| entregue|       24.9|
|      1805|2025-07-31T04:41:29|  cartao_debito|  [{496.67, 112, 2}]|   120867|cancelado|        0.0|
|      3934|2026-05-14T18:05:26| cartao_credito|   [{82.91, 127, 1}]|   129912|     pago|        0.0|
|      2270|2025-12-30T00:00:13| cartao_credito|[{733.29, 162, 1}...|   126879| entregue|        0.0|
|       585|2026-04-28T09:58:39|            pix|[{113.3, 139, 2},...|   126488| en

In [ ]:
clientes.groupBy("estado").count().orderBy("estado", ascending=False).show()

In [ ]:
pedidos.groupBy("status").count().orderBy("status", ascending=False).show()

+---------+-----+
|   status|count|
+---------+-----+
|     pago| 3697|
|  enviado| 3777|
| entregue|15147|
|devolvido| 3776|
|cancelado| 3753|
+---------+-----+



In [ ]:
# criação de um join entre os dataframes clientes e pedidos
join_clientes_pedidos = (pedidos.join(clientes, pedidos.cliente_id == clientes.cliente_id, "inner")) \
                        .select("pedido_id", "forma_pagamento", "nome", "email")

join_clientes_pedidos.show()

+---------+---------------+-----------------+--------------------+
|pedido_id|forma_pagamento|             nome|               email|
+---------+---------------+-----------------+--------------------+
|   123912|            pix|      Felipe Lima|felipe.lima2769@e...|
|   106631|  cartao_debito|      João Araújo|joão.araújo3227@e...|
|   127545|            pix|  Felipe Ferreira|felipe.ferreira45...|
|   120867|  cartao_debito|   Diego Ferreira|diego.ferreira180...|
|   129912| cartao_credito|   Felipe Ribeiro|felipe.ribeiro393...|
|   126879| cartao_credito|       João Silva|joão.silva2270@ex...|
|   126488|            pix|    Camila Araújo|camila.araújo585@...|
|   128719| cartao_credito|    Maria Ribeiro|maria.ribeiro4367...|
|   120550|         boleto| Bruno Cavalcanti|bruno.cavalcanti2...|
|   119070|            pix|  Thiago Oliveira|thiago.oliveira12...|
|   114798|            pix|Fernanda Oliveira|fernanda.oliveira...|
|   122889|            pix|      Bruno Costa|bruno.costa3953@e

Verificando dados da coluna Itens, do df pedidos.

In [ ]:
pedidos = spark.read.json(f"{LANDING}/pedidos.jsonl")
pedidos.printSchema()

root
 |-- cliente_id: long (nullable = true)
 |-- data_pedido: string (nullable = true)
 |-- forma_pagamento: string (nullable = true)
 |-- itens: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- preco_unit: double (nullable = true)
 |    |    |-- produto_id: long (nullable = true)
 |    |    |-- qtd: long (nullable = true)
 |-- pedido_id: long (nullable = true)
 |-- status: string (nullable = true)
 |-- valor_frete: double (nullable = true)



In [ ]:
from pyspark.sql import functions as F

itens = (pedidos
    .select("pedido_id", "cliente_id", "data_pedido", "status",
            F.posexplode("itens").alias("item_seq", "item"))      # posexplode guarda a posição do item
    .select("pedido_id", "cliente_id", "data_pedido", "status", "item_seq",
            F.col("item.produto_id").alias("produto_id"),
            F.col("item.qtd").alias("qtd"),
            F.col("item.preco_unit").alias("preco_unit"))
    .withColumn("valor_item", F.round(F.col("qtd") * F.col("preco_unit"), 2)))

itens.show(8, truncate=False)
print("pedidos:", pedidos.count(), "| itens:", itens.count())    # itens > pedidos

+---------+----------+-------------------+---------+--------+----------+---+----------+----------+
|pedido_id|cliente_id|data_pedido        |status   |item_seq|produto_id|qtd|preco_unit|valor_item|
+---------+----------+-------------------+---------+--------+----------+---+----------+----------+
|123912   |2769      |2025-11-28T17:05:06|devolvido|0       |6         |2  |211.62    |423.24    |
|106631   |3227      |2025-12-22T08:51:13|enviado  |0       |39        |1  |824.72    |824.72    |
|127545   |4575      |2026-02-10T01:37:50|entregue |0       |82        |3  |220.7     |662.1     |
|127545   |4575      |2026-02-10T01:37:50|entregue |1       |62        |3  |704.97    |2114.91   |
|127545   |4575      |2026-02-10T01:37:50|entregue |2       |172       |1  |39.85     |39.85     |
|120867   |1805      |2025-07-31T04:41:29|cancelado|0       |112       |2  |496.67    |993.34    |
|129912   |3934      |2026-05-14T18:05:26|pago     |0       |127       |1  |82.91     |82.91     |
|126879   

In [ ]:
# aplicando checagem de qualidade dos dados
# 1. valores inválidos
itens.filter((F.col("qtd") <= 0) | (F.col("preco_unit") <= 0) | F.col("produto_id").isNull()).count()

# 2. chave composta única (pedido_id, item_seq)
itens.groupBy("pedido_id", "item_seq").count().filter("count > 1").count()

# 3. produto inexistente (integridade referencial)
produtos = spark.read.option("header", True).csv(f"{LANDING}/produtos.csv") \
                .withColumn("produto_id", F.col("produto_id").cast("long"))
itens.join(produtos, "produto_id", "left_anti").count()

0

Salvar os itens pedidos na silver